In [1]:
print("this is a test" + chr(0) + "string")

this is a test string


In [2]:
import re, collections
 
def get_stats(vocab):
    pairs = collections.defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols)-1):
            pairs[symbols[i],symbols[i+1]] += freq # 这里是每两个两个的算频率
    return pairs # 第一次计算时：pairs：defaultdict(<class 'int'>, {('l', 'o'): 7, ('o', 'w'): 7, ('w', '</w>'): 5, ('w', 'e'): 8, ('e', 'r'): 2, ('r', '</w>'): 2, ('n', 'e'): 6, ('e', 'w'): 6, ('e', 's'): 9, ('s', 't'): 9, ('t', '</w>'): 9, ('w', 'i'): 3, ('i', 'd'): 3, ('d', 'e'): 3})
 
 
def merge_vocab(pair, v_in):
    v_out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in v_in:
        w_out = p.sub(''.join(pair), word) # 合并
        v_out[w_out] = v_in[word] # 频率赋值
    return v_out
 
 
vocab = {'l o w </w>': 5, 'l o w e r </w>': 2, 'n e w e s t </w>': 6, 'w i d e s t </w>': 3}
num_merges = 1000
for i in range(num_merges):
    pairs = get_stats(vocab)
    print(f"pairs {i}：{pairs}")
    if not pairs:
        break
    best = max(pairs, key=pairs.get) # ('e', 's')
    vocab = merge_vocab(best, vocab)
    print(best)
    print(f"vocab {i}：{vocab}")

pairs 0：defaultdict(<class 'int'>, {('l', 'o'): 7, ('o', 'w'): 7, ('w', '</w>'): 5, ('w', 'e'): 8, ('e', 'r'): 2, ('r', '</w>'): 2, ('n', 'e'): 6, ('e', 'w'): 6, ('e', 's'): 9, ('s', 't'): 9, ('t', '</w>'): 9, ('w', 'i'): 3, ('i', 'd'): 3, ('d', 'e'): 3})
('e', 's')
vocab 0：{'l o w </w>': 5, 'l o w e r </w>': 2, 'n e w es t </w>': 6, 'w i d es t </w>': 3}
pairs 1：defaultdict(<class 'int'>, {('l', 'o'): 7, ('o', 'w'): 7, ('w', '</w>'): 5, ('w', 'e'): 2, ('e', 'r'): 2, ('r', '</w>'): 2, ('n', 'e'): 6, ('e', 'w'): 6, ('w', 'es'): 6, ('es', 't'): 9, ('t', '</w>'): 9, ('w', 'i'): 3, ('i', 'd'): 3, ('d', 'es'): 3})
('es', 't')
vocab 1：{'l o w </w>': 5, 'l o w e r </w>': 2, 'n e w est </w>': 6, 'w i d est </w>': 3}
pairs 2：defaultdict(<class 'int'>, {('l', 'o'): 7, ('o', 'w'): 7, ('w', '</w>'): 5, ('w', 'e'): 2, ('e', 'r'): 2, ('r', '</w>'): 2, ('n', 'e'): 6, ('e', 'w'): 6, ('w', 'est'): 6, ('est', '</w>'): 9, ('w', 'i'): 3, ('i', 'd'): 3, ('d', 'est'): 3})
('est', '</w>')
vocab 2：{'l o w </w

In [1]:
import re, collections

def get_vocab(filename):
    vocab = collections.defaultdict(int)
    with open(filename, 'r', encoding='utf-8') as fhand:
        for line in fhand:
            words = line.strip().split()
            for word in words:
                vocab[' '.join(list(word)) + ' </w>'] += 1

    return vocab

def get_stats(vocab):
    pairs = collections.defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols)-1):
            pairs[symbols[i],symbols[i+1]] += freq
    return pairs

def merge_vocab(pair, v_in):
    v_out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in v_in:
        w_out = p.sub(''.join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out

def get_tokens_from_vocab(vocab):
    tokens_frequencies = collections.defaultdict(int)
    vocab_tokenization = {}
    for word, freq in vocab.items():
        word_tokens = word.split()
        for token in word_tokens:
            tokens_frequencies[token] += freq
        vocab_tokenization[''.join(word_tokens)] = word_tokens
    return tokens_frequencies, vocab_tokenization

def measure_token_length(token): # 结尾字符不算长度
    if token[-4:] == '</w>':
        return len(token[:-4]) + 1
    else:
        return len(token)

def tokenize_word(string, sorted_tokens, unknown_token='</u>'):
    
    if string == '':
        return []
    if sorted_tokens == []:
        return [unknown_token]

    string_tokens = []
    for i in range(len(sorted_tokens)):
        token = sorted_tokens[i]
        token_reg = re.escape(token.replace('.', '[.]'))

        matched_positions = [(m.start(0), m.end(0)) for m in re.finditer(token_reg, string)]
        if len(matched_positions) == 0:
            continue
        substring_end_positions = [matched_position[0] for matched_position in matched_positions]

        substring_start_position = 0
        for substring_end_position in substring_end_positions:
            substring = string[substring_start_position:substring_end_position]
            string_tokens += tokenize_word(string=substring, sorted_tokens=sorted_tokens[i+1:], unknown_token=unknown_token)
            string_tokens += [token]
            substring_start_position = substring_end_position + len(token)
        remaining_substring = string[substring_start_position:]
        string_tokens += tokenize_word(string=remaining_substring, sorted_tokens=sorted_tokens[i+1:], unknown_token=unknown_token)
        break
    return string_tokens

vocab = {'l o w </w>': 5, 'l o w e r </w>': 2, 'n e w e s t </w>': 6, 'w i d e s t </w>': 3}

# vocab = get_vocab('pg16457.txt')

print('==========')
print('Tokens Before BPE')
tokens_frequencies, vocab_tokenization = get_tokens_from_vocab(vocab) # tokens_frequencies: defaultdict(<class 'int'>, {'l': 7, 'o': 7, 'w': 16, '</w>': 16, 'e': 17, 'r': 2, 'n': 6, 's': 9, 't': 9, 'i': 3, 'd': 3}) vocab_tokenization: {'low</w>': ['l', 'o', 'w', '</w>'], 'lower</w>': ['l', 'o', 'w', 'e', 'r', '</w>'], 'newest</w>': ['n', 'e', 'w', 'e', 's', 't', '</w>'], 'widest</w>': ['w', 'i', 'd', 'e', 's', 't', '</w>']}
print('All tokens: {}'.format(tokens_frequencies.keys()))
print('Number of tokens: {}'.format(len(tokens_frequencies.keys())))
print('==========')

num_merges = 10000
for i in range(num_merges):
    pairs = get_stats(vocab)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)
    print('Iter: {}'.format(i))
    print('Best pair: {}'.format(best))
    tokens_frequencies, vocab_tokenization = get_tokens_from_vocab(vocab)
    print('All tokens: {}'.format(tokens_frequencies.keys()))
    print('Number of tokens: {}'.format(len(tokens_frequencies.keys())))
    print('==========')

# Let's check how tokenization will be for a known word
word_given_known = 'newest</w>lower</w>'
word_given_unknown = 'Ilikeeatingapples!</w>'

sorted_tokens_tuple = sorted(tokens_frequencies.items(), key=lambda item: (measure_token_length(item[0]), item[1]), reverse=True)
sorted_tokens = [token for (token, freq) in sorted_tokens_tuple] # 用来解码

print(sorted_tokens)

word_given = word_given_known 

print('Tokenizing word: {}...'.format(word_given))
if word_given in vocab_tokenization:
    print('Tokenization of the known word:')
    print(vocab_tokenization[word_given])
    print('Tokenization treating the known word as unknown:')
    print(tokenize_word(string=word_given, sorted_tokens=sorted_tokens, unknown_token='</u>'))
else:
    print('Tokenizating of the unknown word:')
    print(tokenize_word(string=word_given, sorted_tokens=sorted_tokens, unknown_token='</u>')) # ['newest</w>', 'lower</w>']

word_given = word_given_unknown 

print('Tokenizing word: {}...'.format(word_given))
if word_given in vocab_tokenization:
    print('Tokenization of the known word:')
    print(vocab_tokenization[word_given])
    print('Tokenization treating the known word as unknown:')
    print(tokenize_word(string=word_given, sorted_tokens=sorted_tokens, unknown_token='</u>'))
else:
    print('Tokenizating of the unknown word:')
    print(tokenize_word(string=word_given, sorted_tokens=sorted_tokens, unknown_token='</u>'))

Tokens Before BPE
All tokens: dict_keys(['l', 'o', 'w', '</w>', 'e', 'r', 'n', 's', 't', 'i', 'd'])
Number of tokens: 11
Iter: 0
Best pair: ('e', 's')
All tokens: dict_keys(['l', 'o', 'w', '</w>', 'e', 'r', 'n', 'es', 't', 'i', 'd'])
Number of tokens: 11
Iter: 1
Best pair: ('es', 't')
All tokens: dict_keys(['l', 'o', 'w', '</w>', 'e', 'r', 'n', 'est', 'i', 'd'])
Number of tokens: 10
Iter: 2
Best pair: ('est', '</w>')
All tokens: dict_keys(['l', 'o', 'w', '</w>', 'e', 'r', 'n', 'est</w>', 'i', 'd'])
Number of tokens: 10
Iter: 3
Best pair: ('l', 'o')
All tokens: dict_keys(['lo', 'w', '</w>', 'e', 'r', 'n', 'est</w>', 'i', 'd'])
Number of tokens: 9
Iter: 4
Best pair: ('lo', 'w')
All tokens: dict_keys(['low', '</w>', 'e', 'r', 'n', 'w', 'est</w>', 'i', 'd'])
Number of tokens: 9
Iter: 5
Best pair: ('n', 'e')
All tokens: dict_keys(['low', '</w>', 'e', 'r', 'ne', 'w', 'est</w>', 'i', 'd'])
Number of tokens: 9
Iter: 6
Best pair: ('ne', 'w')
All tokens: dict_keys(['low', '</w>', 'e', 'r', 'new'

In [3]:
from einops import repeat

i_tensor = torch.randn(3, 224, 224)  
print(i_tensor.shape)
o_tensor = repeat(i_tensor, 'c h w -> n c h w', n=16)  
print(o_tensor.shape)

torch.Size([3, 224, 224])
torch.Size([16, 3, 224, 224])


In [1]:
import torch
import math

# 1. 把图片里的 SGD 类抄下来
class SGD(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3):
        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = {"lr": lr}
        super().__init__(params, defaults)

    def step(self, closure=None):
        loss = None
        if closure is not None:
            loss = closure()

        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                
                # 获取状态 (记录这是第几次 step)
                state = self.state[p]
                t = state.get("t", 0)
                
                grad = p.grad.data
                lr = group["lr"]
                
                # 公式 (20): 学习率随时间 t 衰减
                p.data -= lr / math.sqrt(t + 1) * grad
                
                state["t"] = t + 1
        return loss

# 2. 定义运行实验的辅助函数
def run_experiment(lr):
    print(f"\n--- Testing Learning Rate: {lr} ---")
    # 随机初始化权重 (目标是让它变小)
    weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
    opt = SGD([weights], lr=lr)

    for t in range(10): # 只跑10步
        opt.zero_grad()
        # Loss 函数是权重的平方均值，我们希望它趋近于 0
        loss = (weights**2).mean()
        loss.backward()
        opt.step()
        print(f"Step {t+1}, Loss: {loss.item():.4f}")

# 3. 执行题目要求的三个数值
# 1e1 = 10, 1e2 = 100, 1e3 = 1000
for test_lr in [10.0, 100.0, 1000.0]:
    run_experiment(test_lr)


--- Testing Learning Rate: 10.0 ---
Step 1, Loss: 24.4912
Step 2, Loss: 15.6744
Step 3, Loss: 11.5545
Step 4, Loss: 9.0401
Step 5, Loss: 7.3225
Step 6, Loss: 6.0712
Step 7, Loss: 5.1203
Step 8, Loss: 4.3754
Step 9, Loss: 3.7785
Step 10, Loss: 3.2915

--- Testing Learning Rate: 100.0 ---
Step 1, Loss: 27.5944
Step 2, Loss: 27.5944
Step 3, Loss: 4.7344
Step 4, Loss: 0.1133
Step 5, Loss: 0.0000
Step 6, Loss: 0.0000
Step 7, Loss: 0.0000
Step 8, Loss: 0.0000
Step 9, Loss: 0.0000
Step 10, Loss: 0.0000

--- Testing Learning Rate: 1000.0 ---
Step 1, Loss: 24.3154
Step 2, Loss: 8777.8467
Step 3, Loss: 1516072.1250
Step 4, Loss: 168646864.0000
Step 5, Loss: 13660393472.0000
Step 6, Loss: 862127194112.0000
Step 7, Loss: 44258794930176.0000
Step 8, Loss: 1904202741710848.0000
Step 9, Loss: 70184850857918464.0000
Step 10, Loss: 2253713513673392128.0000
